# Кластеризация водителей по данным до кризиса

Единица наблюдения — клиент. Автомобили, штрафы и заправки сначала агрегируются независимо. В признаки входят только сведения, доступные до 1 июня 2026 года.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'spb_clustering.py').exists():
    ROOT = ROOT / 'hackathon_spb'
sys.path.insert(0, str(ROOT))

from spb_clustering import locate_data_dir, run_pipeline

DATA_DIR = locate_data_dir(None, ROOT)
OUTPUT_DIR = ROOT / 'outputs' / 'clustering'
print('Данные:', DATA_DIR)
print('Результаты:', OUTPUT_DIR)

## Расчёт признаков и кластеров

Полный прогон занимает около минуты и перезаписывает результаты в `outputs/clustering/`.

In [ ]:
result = run_pipeline(DATA_DIR, OUTPUT_DIR)
features = result.features
print(f'Клиентов: {len(features):,}')
print(f'Выбрано кластеров: {result.selected_k}')
display(features.head())

## Выбор числа кластеров

In [ ]:
display(result.diagnostics.style.format({
    'silhouette': '{:.3f}',
    'davies_bouldin': '{:.3f}',
    'stability_ari': '{:.3f}',
    'min_cluster_share': '{:.1%}',
    'max_cluster_share': '{:.1%}',
}))

## Подробные профили

In [ ]:
profile_columns = [
    'cluster_id', 'cluster_label', 'clients', 'share', 'multi_car_share',
    'has_pre_fuel_share', 'has_pre_fine_share', 'mean_cars_count',
    'median_car_value_total', 'mean_car_age_mean',
    'mean_historical_fines_12m_per_car', 'mean_fuel_liters_per_week_pre',
    'mean_fines_per_week_pre',
]
display(result.summary[profile_columns])
display(result.category_profiles)

## Визуализации

In [ ]:
for name in ['cluster_sizes.png', 'cluster_profiles_heatmap.png', 'clusters_pca.png']:
    display(Image(filename=OUTPUT_DIR / 'figures' / name))

## Текстовый отчёт

In [ ]:
display(Markdown((OUTPUT_DIR / 'cluster_report.md').read_text(encoding='utf-8')))